In [755]:
#1 The Data

In [756]:
# Set all seeds to further create reproducible split, negative edges and model:
import torch
import random
import numpy as np

seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

In [757]:
from torch_geometric.datasets import MovieLens100K

In [758]:
dataset = MovieLens100K(root = "./data")

In [759]:
# Data contains 1 graph
len(dataset)

1

In [760]:
# Graph info
# Note that we have a split of 80 000 training positives
# And 20 000 held-out positive test edges
data = dataset[0]
print(data)

HeteroData(
  movie={ x=[1682, 18] },
  user={ x=[943, 24] },
  (user, rates, movie)={
    edge_index=[2, 80000],
    rating=[80000],
    time=[80000],
    edge_label_index=[2, 20000],
    edge_label=[20000],
  },
  (movie, rated_by, user)={
    edge_index=[2, 80000],
    rating=[80000],
    time=[80000],
  }
)


In [761]:
print(data.node_types)

['movie', 'user']


In [762]:
print(data.edge_types)

[('user', 'rates', 'movie'), ('movie', 'rated_by', 'user')]


In [763]:
# Movie vectors represent movie genres (one movie can belong to several genres).
movie_x = data["movie"].x
print(movie_x[:5])

tensor([[0., 0., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.],
        [1., 0., 0., 0., 1., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 1., 0., 1., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.]])


In [764]:
#2 GCN baseline

In [765]:
#2.1. We need to transform the graph, because GCN works with nodes of one type.
#We first transform edge index: shift movie indices by the number of the user nodes,
#and then unite them with user indices in one row.

In [766]:
# The current edge index from the training set (80000 edges: 1st row - users, 2nd row - movies):
edges = data["user", "rates", "movie"].edge_index
print(edges)
print(edges.shape)

tensor([[   0,    0,    0,  ...,  942,  942,  942],
        [   0,    1,    2,  ..., 1187, 1227, 1329]])
torch.Size([2, 80000])


In [767]:
# Take movie indices and shift them by the number of the user nodes
movies = edges[1]
number_users = data["user"].num_nodes
shifted_movies = movies + number_users
users = edges[0]
edges = torch.stack([users, shifted_movies], dim=0)
print(edges)

tensor([[   0,    0,    0,  ...,  942,  942,  942],
        [ 943,  944,  945,  ..., 2130, 2170, 2272]])


In [768]:
# The current pos_edge_index contain a one-directional relation genre -> movie
# Let's reverse the relation and add movie -> genre:
reverse_edges = edges.flip(0) # or one can use "movie, rated_by, user" relation from the initial data
message_passing_edges = torch.cat([edges, reverse_edges], dim=1)
print(message_passing_edges.shape)

torch.Size([2, 160000])


In [769]:
#2.2. Sample negative examples
# We have positive edges (the edges that really exist).
# We now need to sample negative edges (fake edges).
# Then we unite these edges in a new training set,
# so that the recommender predict as mush as possible of the positive (really existing) edges.

In [770]:
number_movies = data["movie"].num_nodes
number_pos = edges.shape[1]
print(number_pos)

80000


In [771]:
# We also need to make sure that none of the 20,000 held-out positive test edges will be sampled as a training negative.
# In other words, we exclude all known interactions when generating negatives.
# Create all known interactions:
all_pos_edges = torch.cat([edges, data["user", "rates", "movie"].edge_label_index], dim=1)
print(all_pos_edges.shape)

torch.Size([2, 100000])


In [772]:
# Sample negative edges excluding all positive edges
from torch_geometric.utils import negative_sampling
random.seed(seed)
neg_edges = negative_sampling(all_pos_edges, (number_users, number_movies), number_pos)
print(neg_edges)
print(neg_edges.shape)

tensor([[ 797,  138,   31,  ...,  416,  626,  124],
        [ 421, 1362,  309,  ..., 1474,  276, 1654]])
torch.Size([2, 80000])


In [773]:
# Then take movie indices for the negative edge index
# and shift them by the number of users (like we did for positive edges)
neg_movies = neg_edges[1]
shifted_neg_movies = neg_movies + number_users
neg_users = neg_edges[0]
neg_edges = torch.stack([neg_users, shifted_neg_movies], dim=0)
print(neg_edges)

tensor([[ 797,  138,   31,  ...,  416,  626,  124],
        [1364, 2305, 1252,  ..., 2417, 1219, 2597]])


In [774]:
# Now we create supervision set (concatenate the pos and neg edges)
# and also create labels 

In [775]:
supervis_edges = torch.cat([edges, neg_edges], dim=1)
print(supervis_edges)
print(supervis_edges.shape)

tensor([[   0,    0,    0,  ...,  416,  626,  124],
        [ 943,  944,  945,  ..., 2417, 1219, 2597]])
torch.Size([2, 160000])


In [776]:
y_1 = torch.ones(edges.shape[1])
y_2 = torch.zeros(neg_edges.shape[1])
y_train = torch.cat([y_1, y_2])
print(y_train.shape)

torch.Size([160000])


In [777]:
#2.4. Create GCN Recommender
# The GCN Recommender first transform the user and movie nodes in a way that they have the same number of features:
# 24 user features -> 64
# 18 user features -> 18
# This is done by linear layers of the GCN recommender.
# Then two GCN layers are stacked

# After the second GCN layer we concatenate the user embedding with the movie embedding, and - via a linear layer -
# get a single prediction whether an edge exists or if it is a fake.

# Note that there are message passing edges and training edges:
# message passing edges are used to build the representation of the nodes
# message training are used to see if the obtained representations perform well to predict true vs false edges

In [778]:
import torch.nn.functional as F
from torch import nn
from torch_geometric.nn import GCNConv

In [779]:
class GCNModel(nn.Module):
    def __init__(self,
                 init_dim_users,
                 init_dim_movies,
                 dim_unified,
                 hidden_dim,
                 embed_dim,
                 prediction_binary):
        super().__init__()
        self.transform_users = nn.Linear(init_dim_users, dim_unified)
        self.transform_movies = nn.Linear(init_dim_movies, dim_unified)
        self.gcn1 = GCNConv(dim_unified, hidden_dim)
        self.gcn2 = GCNConv(hidden_dim, embed_dim)
        self.head = nn.Sequential(nn.Linear(embed_dim * 2, hidden_dim),
                                  nn.ReLU(),
                                  nn.Dropout(0.2),
                                  nn.Linear(hidden_dim, prediction_binary))
    def forward(self, users, movies, message_pass_adj, supervis_adj):
        u = self.transform_users(users)
        m = self.transform_movies(movies)
        x = torch.cat([u, m], dim=0)
        x = self.gcn1.forward(x, message_pass_adj)
        x = F.relu(x)
        x = F.dropout(x, p=0.2, training=self.training)
        x = self.gcn2.forward(x, message_pass_adj)
        x = F.dropout(x, p=0.2, training=self.training)
        x = torch.cat([ x[supervis_adj[0]], x[supervis_adj[1]] ], dim=1)
        x = self.head(x)
        return x

In [780]:
feature_users = data["user"].x.shape[1] #24
feature_movies = data["movie"].x.shape[1] #18

In [781]:
torch.manual_seed(seed)

model = GCNModel(init_dim_users=feature_users, #24
                 init_dim_movies=feature_movies, #18
                 dim_unified=32,
                 hidden_dim=64,
                 embed_dim=32,
                 prediction_binary=1)
optim = torch.optim.Adam(model.parameters(), lr=0.003, weight_decay=1e-5)
loss = nn.BCEWithLogitsLoss()

In [782]:
#2.5. Train GCN Recommender
# Demonstrate binary cross-entropy loss
# as well as the accuracy

In [783]:
torch.manual_seed(seed)
for epoch in range(1501):
    # Predict the training instances, compute the error, backpropagate the error and update the model weights
    model.train()
    optim.zero_grad()
    pred = model.forward(users=data["user"].x, movies=data["movie"].x,
                         message_pass_adj=message_passing_edges, supervis_adj=supervis_edges)
    l = loss(pred.squeeze(1), y_train)
    l.backward()
    optim.step()
    if epoch%20 == 0:
        pred = (pred.sigmoid() >= 0.5).float()
        accuracy = (pred.squeeze(1) == y_train).float().mean()
        print(f"epoch {epoch}, GCN CEL: {l}, GCN accuracy: {accuracy}")

epoch 0, GCN CEL: 0.6931836009025574, GCN accuracy: 0.5114062428474426
epoch 20, GCN CEL: 0.6038931608200073, GCN accuracy: 0.7191062569618225
epoch 40, GCN CEL: 0.4790971279144287, GCN accuracy: 0.7782124876976013
epoch 60, GCN CEL: 0.4503531754016876, GCN accuracy: 0.7952937483787537
epoch 80, GCN CEL: 0.4363154172897339, GCN accuracy: 0.8010125160217285
epoch 100, GCN CEL: 0.43087393045425415, GCN accuracy: 0.8040812611579895
epoch 120, GCN CEL: 0.42632314562797546, GCN accuracy: 0.8063125014305115
epoch 140, GCN CEL: 0.4249282777309418, GCN accuracy: 0.8068125247955322
epoch 160, GCN CEL: 0.4243584871292114, GCN accuracy: 0.8070124983787537
epoch 180, GCN CEL: 0.424862265586853, GCN accuracy: 0.8048999905586243
epoch 200, GCN CEL: 0.41869911551475525, GCN accuracy: 0.809499979019165
epoch 220, GCN CEL: 0.41358378529548645, GCN accuracy: 0.8116687536239624
epoch 240, GCN CEL: 0.4113222658634186, GCN accuracy: 0.8132062554359436
epoch 260, GCN CEL: 0.40648555755615234, GCN accuracy: 

In [784]:
# 2.6 Evaluate GCN Recommender
# First we prepare an evaluation set - in the same way as for the training set

In [785]:
# Shift the indices
test_edges = data["user", "rates", "movie"].edge_label_index # The current edge index for the test data (20000 edges)
test_movies = test_edges[1]
shifted_test_movies = test_movies + number_users
test_users = (data["user", "rates", "movie"].edge_label_index)[0]
test_edges = torch.stack([test_users, shifted_test_movies], dim=0)
print(test_edges)

tensor([[   0,    0,    0,  ...,  458,  459,  461],
        [ 948,  952,  954,  ..., 1876,  952, 1624]])


In [786]:
# Sample negative edges

In [787]:
test_number_pos = test_edges.shape[1]
print(test_number_pos)

20000


In [788]:
random.seed(seed)
test_neg_edges = negative_sampling(
    #note that as before we use all positive edges to sample negatives:
    all_pos_edges,
    (number_users, number_movies),
    test_number_pos)
print(test_neg_edges)
print(test_neg_edges.shape)

tensor([[ 797,  138,   31,  ...,  438,   37,  424],
        [ 421, 1362,  309,  ...,  186,  238,  242]])
torch.Size([2, 20000])


In [789]:
shifted_test_neg_movies = test_neg_edges[1] + number_users
test_neg_users = test_neg_edges[0]
test_neg_edges = torch.stack([test_neg_users, shifted_test_neg_movies], dim=0)
print(test_neg_edges)

tensor([[ 797,  138,   31,  ...,  438,   37,  424],
        [1364, 2305, 1252,  ..., 1129, 1181, 1185]])


In [790]:
# Create test set

In [791]:
test_supervis_edges = torch.cat([test_edges, test_neg_edges], dim=1)
print(test_supervis_edges)
print(test_supervis_edges.shape)

tensor([[   0,    0,    0,  ...,  438,   37,  424],
        [ 948,  952,  954,  ..., 1129, 1181, 1185]])
torch.Size([2, 40000])


In [792]:
y_test_1 = torch.ones(test_edges.shape[1])
y_test_2 = torch.zeros(test_neg_edges.shape[1])
y_test = torch.cat([y_test_1, y_test_2])
print(y_test.shape)

torch.Size([40000])


In [793]:
#2.6.3. Launch evaluation

In [794]:
model.eval()
with torch.no_grad():
    pred_test = model.forward(users=data["user"].x, movies=data["movie"].x,
                              # note that we use message passing edges from the training set
                              message_pass_adj=message_passing_edges,
                              supervis_adj=test_supervis_edges)
    l_test = loss(pred_test.squeeze(1), y_test)
    pred_test = (pred_test.sigmoid() >= 0.5).float()
    accuracy_test = (pred_test.squeeze(1) == y_test).float().mean()
    print(f"GCN test CEL: {l_test}, GCN test accuracy: {accuracy_test}")

GCN test CEL: 0.434054970741272, GCN test accuracy: 0.808525025844574


In [795]:
#3 RCGN Recommender
# We use 5 movie ratings as edge type
# We create 18 genres

In [796]:
#3.1. Create genre edges

In [797]:
# Since there is no edge relation in the initial data, we need to create genre edges first.
# We do that by reconstructing information about genres from movie embeddings.
# It is possible because the features of the movie embeddings represent genres:
# the embedding has 18 features (0 and 1) such that 1 stands for "the movie has this genre":

print(movie_x[:5])

tensor([[0., 0., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.],
        [1., 0., 0., 0., 1., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 1., 0., 1., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.]])


In [798]:
# With torch nonzero we define the exact place of each 1 in movie_x:
movie_ids, genre_ids = movie_x.nonzero(as_tuple=True)
# Now we can create a new edge type in the data (genre -> movie)
edges_genre_movie = torch.stack([genre_ids, movie_ids], dim=0)
print(edges_genre_movie)
print(edges_genre_movie.shape)

tensor([[   2,    3,    4,  ...,   13,    4,    7],
        [   0,    0,    0,  ..., 1679, 1680, 1681]])
torch.Size([2, 2891])


In [799]:
# create the reverse relation - from movies to genres (movie -> genre)
edges_movie_genre = edges_genre_movie.flip(0)
print(edges_movie_genre)
print(edges_movie_genre.shape)

tensor([[   0,    0,    0,  ..., 1679, 1680, 1681],
        [   2,    3,    4,  ...,   13,    4,    7]])
torch.Size([2, 2891])


In [800]:
# all genres are already in genre_edges (the first row),
# hence all genre-to-movie edges can use one semantic relation:
# genre --describes--> movie

genre_relation = torch.zeros(edges_genre_movie.size(1), dtype=torch.long) # 1 relation
print(genre_relation)
print(genre_relation.shape)

tensor([0, 0, 0,  ..., 0, 0, 0])
torch.Size([2891])


In [801]:
#3.2 Create genre nodes
# Note that we do not have genre nodes
# So we create them by initializing embedding direct in the RGCN Recommender
# We initialize 18 embeddings in the RGCN Recommender and they are learned during training

In [802]:
num_genres = data["movie"].x.size(1)
print(num_genres)

18


In [803]:
num_genres = torch.arange(num_genres) # We trainsform the integer 18 into the tensor,
# because the tensor-format is needed for initializing embeddings via nn.Embedding
print(num_genres)

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17])


In [804]:
# 3.3 Create message_passing user-movie edges 
rgcn_edges_user_movie = data["user", "rates", "movie"].edge_index
rgcn_edges_movie_user = rgcn_edges_user_movie.flip(0)
print(rgcn_edges_user_movie)
print(rgcn_edges_movie_user)

tensor([[   0,    0,    0,  ...,  942,  942,  942],
        [   0,    1,    2,  ..., 1187, 1227, 1329]])
tensor([[   0,    1,    2,  ..., 1187, 1227, 1329],
        [   0,    0,    0,  ...,  942,  942,  942]])


In [805]:
user_relations = data["user", "rates", "movie"].rating.long() - 1
print(user_relations) # 5 relations

tensor([4, 2, 3,  ..., 2, 2, 2])


In [806]:
# 3.4. Create RGCN supervision edges
# We can obtain them from GCN supervision edges by simply shifting movie indices back
print(supervis_edges)
print(supervis_edges.shape)

tensor([[   0,    0,    0,  ...,  416,  626,  124],
        [ 943,  944,  945,  ..., 2417, 1219, 2597]])
torch.Size([2, 160000])


In [807]:
rgcn_supervis_edges = torch.stack([supervis_edges[0], supervis_edges[1] - 943])
print(rgcn_supervis_edges)
print(rgcn_supervis_edges.shape)

tensor([[   0,    0,    0,  ...,  416,  626,  124],
        [   0,    1,    2,  ..., 1474,  276, 1654]])
torch.Size([2, 160000])


In [808]:
y_train_rgcn = y_train
print(y_train_rgcn.shape)

torch.Size([160000])


In [809]:
#4.3 Create RGCN-genre Recommender
# Genre embeddings are initialized via nn.Embedding

# There are 7 RGCN layers: 1, 2, 3, 4, 5, 6, 7

# RGCN layers 1 and 2 give us two types of movie representations:
# the first one is got from "user -> movie" relation,
# the other is formed via "genre -> movie" relation
# We then sum up these two representations (they have the same size)
# So we get a single movie representation

# Level 5 builds user representation.
# Level 7 builds genre representation.

# Having the movie representation (from levels 1 and 2), on the one hand,
# and the user and genre representaion (from levels 5 and 7)
# we use all these representations for RGCN layers 3 and 4.
# As a result, we get the a movie embedding.

# Layer 6 finalizes a user embedding.

# Finally, we concatenate the user embedding with the movie embedding,
# and get a prediction, whether the edge exists or it is a fake.

In [810]:
from torch_geometric.nn import RGCNConv

In [811]:
class RGCNModel(nn.Module):
    def __init__(self, dim_users,
                 dim_movies,
                 dim_genres,
                 hidden_dim,
                 embed_dim, 
                 prediction_binary):
        super().__init__()
        # build movie representations:
        self.genre = torch.nn.Embedding(num_embeddings=dim_movies, embedding_dim=dim_genres)
        self.rgcn1 = RGCNConv((dim_users, dim_movies), hidden_dim, num_relations=5)
        self.rgcn2 = RGCNConv((dim_genres, dim_movies), hidden_dim, num_relations=1)
        self.rgcn3 = RGCNConv((hidden_dim, hidden_dim), embed_dim, num_relations=5)
        self.rgcn4 = RGCNConv((hidden_dim, hidden_dim), embed_dim, num_relations=1)
        # build user representations:
        self.rgcn5 = RGCNConv((dim_movies, dim_users), hidden_dim, num_relations=5)
        self.rgcn6 = RGCNConv((dim_movies, hidden_dim), embed_dim, num_relations=5)
        self.head = nn.Sequential(nn.Linear(embed_dim * 2, hidden_dim),
                                  nn.ReLU(),
                                  nn.Dropout(0.2),
                                  nn.Linear(hidden_dim, prediction_binary))
        # build genre representations:
        self.rgcn7 = RGCNConv((dim_movies, dim_genres), hidden_dim, num_relations=1)

    def forward(self,
                users, movies, genres,
                message_pass_adj_user_movie, message_pass_adj_movie_user, user_relations,
                message_pass_adj_genre_movie, message_pass_adj_movie_genre, genre_relation,
                supervis_adj):
        # build movie representations (1st iteration):
        m_emb_1 = self.rgcn1.forward((users, movies), message_pass_adj_user_movie, user_relations)
        genres = self.genre(genres)
        m_emb_2 = self.rgcn2.forward((genres, movies), message_pass_adj_genre_movie, genre_relation)
        m_emb = m_emb_1 + m_emb_2
        m_emb = F.relu(m_emb)
        m_emb = F.dropout(m_emb, p=0.2, training=self.training)

        # build user representations (1st iteration)
        u_emb = self.rgcn5((movies, users), message_pass_adj_movie_user, user_relations)
        u_emb = F.relu(u_emb)
        u_emb = F.dropout(u_emb, p=0.2, training=self.training)

        # build genre representations (1st iteration)
        g_emb = self.rgcn7.forward((movies, genres), message_pass_adj_movie_genre, genre_relation)
        g_emb = F.relu(g_emb)
        g_emb = F.dropout(g_emb, p=0.2, training=self.training)

        # build movie representations (2nd iteration):
        m_emb_1 = self.rgcn3.forward((u_emb, m_emb), message_pass_adj_user_movie, user_relations)
        m_emb_2 = self.rgcn4.forward((g_emb, m_emb), message_pass_adj_genre_movie, genre_relation)
        m_emb = m_emb_1 + m_emb_2
        m_emb = F.dropout(m_emb, p=0.2, training=self.training)
        
        # build users representation (2nd iteration):
        u_emb = self.rgcn6((movies, u_emb), message_pass_adj_movie_user, user_relations)
        u_emb = F.dropout(u_emb, p=0.2, training=self.training)
        
        # concatenate user representation (that we obtained after two iterations)
        # and movie representation (that we obtained after two iterations)
        # and predict if the edge is real or a fake
        x = torch.cat([ u_emb[supervis_adj[0]], m_emb[supervis_adj[1]] ], dim=1)
        x = self.head(x)
        return x

In [812]:
torch.manual_seed(seed)
model_rgcn = RGCNModel(dim_users=feature_users, #24
                             dim_movies=feature_movies, #18
                             dim_genres=16,
                             hidden_dim=64,
                             embed_dim=32,
                             prediction_binary=1)
optim = torch.optim.Adam(model_rgcn.parameters(), lr=0.003, weight_decay=1e-5)
loss = nn.BCEWithLogitsLoss()

In [813]:
#4.4. Train RGCN-genre recommender

In [814]:
torch.manual_seed(seed)
for epoch in range(1501):
    # Predict the training instances, compute the error, backpropagate the error and update the model weights
    model_rgcn.train()
    optim.zero_grad()
    pred_rgcn = model_rgcn.forward(users=data["user"].x, movies=data["movie"].x, genres=num_genres,
                                   message_pass_adj_user_movie = rgcn_edges_user_movie,
                                   message_pass_adj_movie_user = rgcn_edges_movie_user,
                                   user_relations = user_relations,
                                   message_pass_adj_genre_movie = edges_genre_movie,
                                   message_pass_adj_movie_genre = edges_movie_genre,
                                   genre_relation = genre_relation,
                                   supervis_adj=rgcn_supervis_edges)
    l_rgcn = loss(pred_rgcn.squeeze(1), y_train_rgcn)
    l_rgcn.backward()
    optim.step()
    if epoch%20 == 0:
        pred_rgcn = (pred_rgcn.sigmoid() >= 0.5).float()
        accuracy_rgcn = (pred_rgcn.squeeze(1) == y_train).float().mean()
        print(f"epoch {epoch}, RCGN CEL: {l_rgcn}, RGCN accuracy: {accuracy_rgcn}")

epoch 0, RCGN CEL: 0.6934011578559875, RGCN accuracy: 0.5163687467575073
epoch 20, RCGN CEL: 0.5390921235084534, RGCN accuracy: 0.7275500297546387
epoch 40, RCGN CEL: 0.5005532503128052, RGCN accuracy: 0.7560437321662903
epoch 60, RCGN CEL: 0.4737784266471863, RGCN accuracy: 0.7764437794685364
epoch 80, RCGN CEL: 0.45315343141555786, RGCN accuracy: 0.7898562550544739
epoch 100, RCGN CEL: 0.43930721282958984, RGCN accuracy: 0.7967625260353088
epoch 120, RCGN CEL: 0.4234146475791931, RGCN accuracy: 0.8083062767982483
epoch 140, RCGN CEL: 0.4135289490222931, RGCN accuracy: 0.8126749992370605
epoch 160, RCGN CEL: 0.408595472574234, RGCN accuracy: 0.8149437308311462
epoch 180, RCGN CEL: 0.4013068974018097, RGCN accuracy: 0.8198999762535095
epoch 200, RCGN CEL: 0.39949867129325867, RGCN accuracy: 0.8209249973297119
epoch 220, RCGN CEL: 0.3946612775325775, RGCN accuracy: 0.8238687515258789
epoch 240, RCGN CEL: 0.39057374000549316, RGCN accuracy: 0.8263499736785889
epoch 260, RCGN CEL: 0.38813

In [815]:
#4.5. Evaluate RGCN-genre recommender

In [816]:
# 3.4. Create RGCN test supervision edges
# We can obtain them from GCN test supervision edges by simply shifting movie indices back
print(test_supervis_edges)
print(test_supervis_edges.shape)

tensor([[   0,    0,    0,  ...,  438,   37,  424],
        [ 948,  952,  954,  ..., 1129, 1181, 1185]])
torch.Size([2, 40000])


In [817]:
rgcn_test_supervis_edges = torch.stack([test_supervis_edges[0], test_supervis_edges[1] - 943])
print(rgcn_test_supervis_edges)
print(rgcn_test_supervis_edges.shape)

tensor([[  0,   0,   0,  ..., 438,  37, 424],
        [  5,   9,  11,  ..., 186, 238, 242]])
torch.Size([2, 40000])


In [818]:
y_test_rgcn = y_test
print(y_test_rgcn.shape)

torch.Size([40000])


In [819]:
model_rgcn.eval()
with torch.no_grad():
    pred_rgcn_test = model_rgcn.forward(users=data["user"].x, movies=data["movie"].x, genres=num_genres,
                                        message_pass_adj_user_movie = rgcn_edges_user_movie,
                                        message_pass_adj_movie_user = rgcn_edges_movie_user,
                                        user_relations = user_relations,
                                        message_pass_adj_genre_movie = edges_genre_movie,
                                        message_pass_adj_movie_genre = edges_movie_genre,
                                        genre_relation = genre_relation,
                                        supervis_adj=rgcn_test_supervis_edges)
    l_test_rgcn = loss(pred_rgcn_test.squeeze(1), y_test_rgcn)
    pred_rgcn_test = (pred_rgcn_test.sigmoid() >= 0.5).float()
    accuracy_test_rgcn = (pred_rgcn_test.squeeze(1) == y_test).float().mean()
    print(f"RGCN test CEL: {l_test_rgcn}, RGCN test accuracy: {accuracy_test_rgcn}")

RGCN test CEL: 0.36681053042411804, RGCN test accuracy: 0.8434000015258789
